In [2]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import random

# Parameters
frame_length = 4  # seconds
overlap = 1  # seconds
sr = 16000  # sample rate
n_mfcc = 39  # Number of MFCC coefficients
codebook_size = 100  # BoAW codebook size

def remove_silence_and_noise(audio_path, sr):
    """Load audio and remove silence and noise."""
    audio, sr = librosa.load(audio_path, sr=sr)
    intervals = librosa.effects.split(audio, top_db=20)  # Adjust top_db as needed
    clean_audio = np.concatenate([audio[start:end] for start, end in intervals])
    return clean_audio

def segment_audio(audio, sr, frame_length, overlap):
    """Segment audio into fixed-length frames with overlap."""
    step = int((frame_length - overlap) * sr)
    segments = [audio[i:i + frame_length * sr] for i in range(0, len(audio), step) if len(audio[i:i + frame_length * sr]) == frame_length * sr]
    return segments

def extract_mfcc_features(segments, sr, n_mfcc):
    """Extract MFCC features from audio segments."""
    mfcc_features = []
    for segment in segments:
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfcc, axis=1)  # Aggregate over time
        mfcc_features.append(mfcc_mean)
    return np.array(mfcc_features)

def create_boaw_features(mfcc_features, codebook_size):
    """Convert MFCC features into Bag-of-Audio-Words (BoAW) representation."""
    kmeans = KMeans(n_clusters=codebook_size, random_state=42)
    kmeans.fit(mfcc_features.astype(np.float64))  # Ensure data type is float64
    boaw_features = []
    for feature in mfcc_features:
        histogram = np.histogram(kmeans.predict([feature.astype(np.float64)]), bins=np.arange(codebook_size + 1))[0]
        boaw_features.append(histogram)
    return np.array(boaw_features)


def save_features_to_csv(boaw_features, output_path, labels):
    """Save BoAW features and labels to a CSV file."""
    df = pd.DataFrame(boaw_features)
    df['label'] = labels
    df.to_csv(output_path, index=False)

def main(input_dirs, output_csv):
    """Main function to process audio files and save features."""
    all_boaw_features = []
    labels = []

    for input_directory in input_dirs:
        label = 0 if 'HC' in input_directory else 1  # Assign labels: HC -> 0, MDD -> 1
        for root, _, files in os.walk(input_directory):
            for file in files:
                if file.endswith(".wav"):
                    file_path = os.path.join(root, file)
                    print(f"Processing {file_path}")

                    # Load and preprocess audio
                    clean_audio = remove_silence_and_noise(file_path, sr)

                    # Segment audio
                    segments = segment_audio(clean_audio, sr, frame_length, overlap)

                    # Extract MFCC features
                    mfcc_features = extract_mfcc_features(segments, sr, n_mfcc)

                    # Append features and labels for BoAW processing
                    if len(mfcc_features) > 0:
                        all_boaw_features.extend(mfcc_features)
                        labels.extend([label] * len(mfcc_features))

    # Convert MFCC features to BoAW representation
    print("Generating BoAW features...")
    all_boaw_features = np.array(all_boaw_features)
    boaw_features = create_boaw_features(all_boaw_features, codebook_size)

    # Save to CSV
    print(f"Saving features to {output_csv}")
    save_features_to_csv(boaw_features, output_csv, labels)

if __name__ == "__main__":
    input_dirs = [
        r"C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female",
        r"C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Male",
        r"C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_MDD\Female",
        r"C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_MDD\Male"
    ]
    output_csv = "boaw_features.csv"  # 
    main(input_dirs, output_csv)


Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\303_P\303_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\304_P\304_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\306_P\306_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\307_P\307_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\314_P\314_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\320_P\320_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\323_P\323_AUDIO.wav
Processing C:\Users\Administrator\PycharmProjects\Edaic\Processed_Audios\interview_removed_HC\Female\325_P\325_AUDIO.wav
Processing C:\Users\Administrato